# Driver Positioning Pipeline — dbt Version
Stages 1–5 are now handled by dbt (DuckDB). This notebook starts directly from the feature matrix.

In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
import duckdb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from geopy.distance import geodesic
from itertools import product
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('All imports OK')

All imports OK


## 1. Load Feature Matrix from DuckDB (replaces Stages 1–5)

In [5]:
# ── Read feature matrix built by dbt ──────────────────────────────────────────
con = duckdb.connect(r"C:\Users\pujit\DE PROJECT\pipeline.duckdb")
demand_df = con.execute("SELECT * FROM fct_feature_matrix").df()
con.close()

# Rename column to match original notebook
demand_df = demand_df.rename(columns={'PULocationID': 'PULocationID', 'time_window': 'time_window'})

print(f'Loaded {len(demand_df):,} rows from DuckDB')
print(f'Columns: {list(demand_df.columns)}')
print(demand_df.head(3).to_string(index=False))

Loaded 763,475 rows from DuckDB
Columns: ['trip_count', 'PULocationID', 'time_window', 'avg_fare_usd', 'per_driver_revenue_usd', 'est_drivers', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_96', 'rolling_mean_4', 'rolling_std_4', 'rolling_mean_96']
 trip_count  PULocationID         time_window  avg_fare_usd  per_driver_revenue_usd  est_drivers  hour_sin  hour_cos  weekday_sin  weekday_cos  month_sin    month_cos  lag_1  lag_2  lag_4  lag_8  lag_96  rolling_mean_4  rolling_std_4  rolling_mean_96
         32            48 2025-03-11 07:45:00     11.785625                8.770698         43.0  0.965926 -0.258819     0.974928    -0.222521        1.0 6.123234e-17     20     25     28     12      27            23.0       4.242641        21.072917
         31            48 2025-03-11 08:00:00     14.800000               10.923810         42.0  0.866025 -0.500000     0.974928    -0.222521        1.0 6.123234e-17     32 

## 2. Also load raw cleaned data for fare/zone stats used in WSM

In [6]:
# ── Load fare averages per zone from the cleaned intermediate table ────────────
con = duckdb.connect(r"C:\Users\pujit\DE PROJECT\pipeline.duckdb")
fare_by_zone = con.execute("""
    SELECT zone_id AS PULocationID, AVG(fare_amount_usd) AS avg_fare
    FROM int_tlc_cleaned
    GROUP BY zone_id
""").df()
con.close()

print(f'Fare stats loaded for {len(fare_by_zone):,} zones')

Fare stats loaded for 258 zones


## 3. Train / Test Split (Temporal — No Leakage)

In [7]:
# ── Strict temporal split ──────────────────────────────────────────────────────
demand_df = demand_df.sort_values(['PULocationID', 'time_window'])
demand_df['time_window'] = pd.to_datetime(demand_df['time_window'])

split_date = demand_df['time_window'].quantile(0.8)
print(f'Train up to : {split_date}')
print(f'Test  from  : {split_date}')

train = demand_df[demand_df['time_window'] <= split_date]
test  = demand_df[demand_df['time_window'] >  split_date]

features = [
    'PULocationID',
    'hour_sin', 'hour_cos',
    'weekday_sin', 'weekday_cos',
    'month_sin', 'month_cos',
    'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_96',
    'rolling_mean_4', 'rolling_std_4', 'rolling_mean_96'
]

X_train, y_train = train[features], train['trip_count']
X_test,  y_test  = test[features],  test['trip_count']

print(f'Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}')

Train up to : 2025-03-16 13:45:00
Test  from  : 2025-03-16 13:45:00
Train size: 610,886 | Test size: 152,589


## 4. Train XGBoost Model

In [8]:
# ── Train ─────────────────────────────────────────────────────────────────────
model = xgb.XGBRegressor(
    objective        = 'reg:squarederror',
    n_estimators     = 800,
    learning_rate    = 0.02,
    max_depth        = 8,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 5,
    random_state     = 42,
    n_jobs           = -1
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)
print('Training complete')

[0]	validation_0-rmse:20.36464
[100]	validation_0-rmse:5.93044
[200]	validation_0-rmse:5.20895
[300]	validation_0-rmse:5.13350
[400]	validation_0-rmse:5.11394
[500]	validation_0-rmse:5.10117
[600]	validation_0-rmse:5.08957
[700]	validation_0-rmse:5.08318
[799]	validation_0-rmse:5.07569
Training complete


## 5. Evaluation — Full Metrics

In [9]:
# ── Predict & evaluate ────────────────────────────────────────────────────────
y_pred = model.predict(X_test)
y_pred = np.clip(y_pred, 0, None)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

mask = y_test > 0
mape  = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100
nrmse = rmse / (y_test.max() - y_test.min()) * 100

test_res = test[['PULocationID', 'time_window']].copy()
test_res['actual']    = y_test.values
test_res['predicted'] = y_pred

print('=' * 45)
print('        MODEL EVALUATION METRICS')
print('=' * 45)
print(f'  MAE              : {mae:.4f}')
print(f'  RMSE             : {rmse:.4f}')
print(f'  R²               : {r2:.4f}')
print(f'  MAPE             : {mape:.2f}%')
print(f'  Normalized RMSE  : {nrmse:.2f}%')
print(f'  Demand range     : {int(y_test.min())} – {int(y_test.max())}')
print('=' * 45)

        MODEL EVALUATION METRICS
  MAE              : 2.6004
  RMSE             : 5.0757
  R²               : 0.9399
  MAPE             : 41.05%
  Normalized RMSE  : 1.97%
  Demand range     : 1 – 259


## 6. WSM Setup with Driver Supply

In [10]:
# ── Zone stats from test set ──────────────────────────────────────────────────
zone_stats = test_res.groupby('PULocationID').agg(
    predicted_demand = ('predicted', 'mean'),
    actual_demand    = ('actual',    'mean')
).reset_index()

# Demand density and driver supply from dbt feature matrix
density = demand_df.groupby('PULocationID')['trip_count'].mean().reset_index()
density.columns = ['PULocationID', 'demand_density']

supply = demand_df.groupby('PULocationID')['est_drivers'].mean().reset_index()
supply.columns = ['PULocationID', 'avg_drivers']

ZONE_COORDS = {
    161: (40.7589, -73.9851), 162: (40.7614, -73.9776),
    163: (40.7549, -73.9840), 164: (40.7484, -73.9967),
    230: (40.7527, -73.9772), 237: (40.7736, -73.9566),
    236: (40.7685, -73.9566), 186: (40.7282, -74.0776),
    132: (40.6413, -73.7781), 138: (40.7769, -73.8740),
    229: (40.7549, -73.9840),
}
DEFAULT_COORD = (40.7580, -73.9855)

zones = zone_stats \
    .merge(fare_by_zone, on='PULocationID', how='left') \
    .merge(density,      on='PULocationID', how='left') \
    .merge(supply,       on='PULocationID', how='left')

zones['lat'] = zones['PULocationID'].map(lambda z: ZONE_COORDS.get(z, DEFAULT_COORD)[0])
zones['lon'] = zones['PULocationID'].map(lambda z: ZONE_COORDS.get(z, DEFAULT_COORD)[1])
zones['predicted_demand']    = zones['predicted_demand'].clip(lower=0)
zones['supply_demand_ratio'] = zones['avg_drivers'] / (zones['demand_density'] + 1e-6)

print(f'Zones available for WSM scoring: {len(zones)}')
print(zones[['PULocationID','predicted_demand','avg_fare',
             'demand_density','avg_drivers','supply_demand_ratio']]
      .sort_values('predicted_demand', ascending=False)
      .head(10).to_string(index=False))

Zones available for WSM scoring: 232
 PULocationID  predicted_demand  avg_fare  demand_density  avg_drivers  supply_demand_ratio
          161         60.179817 15.506349       58.606079    78.486558             1.339222
          237         55.044228 12.592348       57.959882    77.626015             1.339306
          132         53.651779 62.868205       46.962557    62.953255             1.340499
          236         49.454140 13.179939       55.230301    73.992006             1.339699
          230         43.886967 17.831469       41.292074    55.393358             1.341501
          186         43.225975 15.729092       41.165517    55.219976             1.341413
          162         43.044651 15.077839       41.657811    55.888252             1.341603
          138         42.978420 42.621077       38.134225    51.199155             1.342604
          142         39.050156 13.958263       39.002857    52.351385             1.342245
          234         37.463242 14.048512  

## 7. WSM Weight Grid Search

In [16]:
# ── Grid search over WSM weights ──────────────────────────────────────────────
FUEL_RATE  = 0.12
driver_loc = (40.7580, -73.9855)#MANHATTAN 

alpha_vals = [0.6, 0.8, 1.0]
beta_vals  = [0.2, 0.4, 0.6]
gamma_vals = [0.1, 0.2, 0.3]

best_score   = -np.inf
best_weights = (1.0, 0.5, 0.3)

def compute_wsm(driver_loc, zone_row, alpha, beta, gamma):
    zone_loc   = (zone_row['lat'], zone_row['lon'])
    distance   = geodesic(driver_loc, zone_loc).km
    revenue    = (zone_row['predicted_demand'] * zone_row['avg_fare']) / \
                  max(zone_row['avg_drivers'], 1)
    idle_pen   = 1 / (zone_row['demand_density'] + 1e-6)
    supply_pen = zone_row['supply_demand_ratio']
    fuel_cost  = distance * FUEL_RATE
    return alpha * revenue - beta * (idle_pen + supply_pen) - gamma * fuel_cost

for a, b, g in product(alpha_vals, beta_vals, gamma_vals):
    zones['wsm_tmp'] = zones.apply(lambda z: compute_wsm(driver_loc, z, a, b, g), axis=1)
    best_z = zones.sort_values('wsm_tmp', ascending=False).iloc[0]
    score  = best_z['predicted_demand'] * best_z['avg_fare'] / max(best_z['avg_drivers'], 1)
    if score > best_score:
        best_score   = score
        best_weights = (a, b, g)

alpha, beta, gamma = best_weights
print(f'Best WSM weights  →  α={alpha}, β={beta}, γ={gamma}')
print(f'Best per-driver revenue score: ${best_score:.2f}')

Best WSM weights  →  α=0.6, β=0.2, γ=0.1
Best per-driver revenue score: $53.58


## 8. Final WSM Scoring & Recommendation

In [12]:
# ── Compute final WSM scores ──────────────────────────────────────────────────
zones['WSM_score'] = zones.apply(
    lambda z: compute_wsm(driver_loc, z, alpha, beta, gamma), axis=1
)
zones_sorted = zones.sort_values('WSM_score', ascending=False)

print('\nTop 10 Zones by WSM Score:')
print(zones_sorted[['PULocationID','predicted_demand','avg_fare',
                     'avg_drivers','supply_demand_ratio','WSM_score']]
      .head(10).to_string(index=False))

best_zone      = zones_sorted.iloc[0]
per_driver_rev = (best_zone['predicted_demand'] * best_zone['avg_fare']) / \
                  max(best_zone['avg_drivers'], 1)

print(f"\n{'='*52}")
print(f"           RECOMMENDED ZONE")
print(f"{'='*52}")
print(f"  Zone (PULocationID) : {int(best_zone['PULocationID'])}")
print(f"  Predicted Demand    : {best_zone['predicted_demand']:.2f} trips/15min")
print(f"  Avg Fare            : ${best_zone['avg_fare']:.2f}")
print(f"  Est. Drivers        : {best_zone['avg_drivers']:.1f}")
print(f"  Supply/Demand Ratio : {best_zone['supply_demand_ratio']:.4f}")
print(f"  Per-Driver Revenue  : ${per_driver_rev:.2f} per window")
print(f"  WSM Score           : {best_zone['WSM_score']:.4f}")
print(f"{'='*52}")


Top 10 Zones by WSM Score:
 PULocationID  predicted_demand  avg_fare  avg_drivers  supply_demand_ratio  WSM_score
          132         53.651779 62.868205    62.953255             1.340499  31.613631
            1          1.113331 75.896792     2.028846             1.971961  24.400271
          138         42.978420 42.621077    51.199155             1.342604  21.077097
           93          1.146747 61.766959     2.128408             1.886203  19.412853
           70          5.047688 41.700231     6.473726             1.413596  19.182290
           10          1.456170 47.099674     2.413393             1.736906  16.559812
          219          1.258620 49.725196     2.200106             1.842174  16.531952
          117          1.265354 45.577728     2.162338             1.863308  15.457663
          195          1.919974 40.222745     2.926410             1.651785  15.390480
          207          1.103109 48.315085     2.037313             1.964027  15.110584

           REC

## 9. Baseline Comparison

In [13]:
# ── Baseline 1: Random zone ───────────────────────────────────────────────────
random_rev = (zones['predicted_demand'] * zones['avg_fare'] / zones['avg_drivers']).mean()

# ── Baseline 2: Nearest zone ──────────────────────────────────────────────────
zones['dist_km'] = zones.apply(lambda z: geodesic(driver_loc, (z['lat'], z['lon'])).km, axis=1)
nearest_zone = zones.sort_values('dist_km').iloc[0]
nearest_rev  = (nearest_zone['predicted_demand'] * nearest_zone['avg_fare']) / \
                max(nearest_zone['avg_drivers'], 1)

# ── Baseline 3: Greedy demand ─────────────────────────────────────────────────
greedy_zone = zones.sort_values('predicted_demand', ascending=False).iloc[0]
greedy_rev  = (greedy_zone['predicted_demand'] * greedy_zone['avg_fare']) / \
               max(greedy_zone['avg_drivers'], 1)

our_rev = per_driver_rev

print('='*55)
print('       BASELINE COMPARISON (Per-Driver Revenue)')
print('='*55)
print(f'  Random Zone              : ${random_rev:.2f} / 15-min window')
print(f'  Nearest Zone             : ${nearest_rev:.2f} / 15-min window')
print(f'  Greedy Demand            : ${greedy_rev:.2f} / 15-min window')
print(f'  Our System (WSM)         : ${our_rev:.2f} / 15-min window')
print('-'*55)
print(f'  Improvement vs Random    : {((our_rev-random_rev)/random_rev)*100:.1f}%')
print(f'  Improvement vs Nearest   : {((our_rev-nearest_rev)/nearest_rev)*100:.1f}%')
print(f'  Improvement vs Greedy    : {((our_rev-greedy_rev)/greedy_rev)*100:.1f}%')
print('='*55)

       BASELINE COMPARISON (Per-Driver Revenue)
  Random Zone              : $16.79 / 15-min window
  Nearest Zone             : $41.65 / 15-min window
  Greedy Demand            : $11.89 / 15-min window
  Our System (WSM)         : $53.58 / 15-min window
-------------------------------------------------------
  Improvement vs Random    : 219.2%
  Improvement vs Nearest   : 28.6%
  Improvement vs Greedy    : 350.6%


## 10. Graphs

In [14]:
# ── Graph 1: Actual vs Predicted ──────────────────────────────────────────────
sample_idx = np.random.choice(len(y_test), min(3000, len(y_test)), replace=False)
y_t_s = y_test.values[sample_idx]
y_p_s = y_pred[sample_idx]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(y_t_s, y_p_s, alpha=0.3, s=12, color='steelblue', edgecolors='none')
mn, mx = y_test.min(), y_test.max()
ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect Prediction')
ax.set_xlabel('Actual Demand')
ax.set_ylabel('Predicted Demand')
ax.set_title(f'Actual vs Predicted Demand\n$R^2$={r2:.4f}, MAPE={mape:.1f}%')
ax.legend()
plt.tight_layout()
plt.savefig('graph1_actual_vs_predicted.png', dpi=150)
plt.show()
print('Graph 1 saved')

Graph 1 saved


In [12]:
# ── Graph 2: Residual Distribution ───────────────────────────────────────────
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals, bins=60, color='steelblue', edgecolor='black', linewidth=0.3)
axes[0].axvline(0, color='red', linestyle='--', lw=1.5)
axes[0].set_xlabel('Residual (Actual − Predicted)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Residual Distribution')

axes[1].scatter(y_pred[sample_idx], residuals[sample_idx], alpha=0.3, s=10, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted Demand')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual vs Predicted')

plt.tight_layout()
plt.savefig('graph2_residuals.png', dpi=150)
plt.show()
print('Graph 2 saved')

Graph 2 saved


In [13]:
# ── Graph 3: Feature Importance ───────────────────────────────────────────────
imp_df = pd.DataFrame({
    'feature':    features,
    'importance': model.feature_importances_
}).sort_values('importance')

plt.figure(figsize=(7, 5))
plt.barh(imp_df['feature'], imp_df['importance'],
         color='steelblue', edgecolor='black', linewidth=0.4)
plt.xlabel('Importance Score')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('graph3_feature_importance.png', dpi=150)
plt.show()
print('Graph 3 saved')

Graph 3 saved


In [14]:
# ── Graph 4: WSM Score — Top 20 Zones ─────────────────────────────────────────
top20  = zones_sorted.head(20)
colors = ['green' if i == 0 else 'steelblue' for i in range(len(top20))]

plt.figure(figsize=(10, 5))
plt.bar(top20['PULocationID'].astype(str), top20['WSM_score'],
        color=colors, edgecolor='black', linewidth=0.4)
plt.xlabel('Zone (PULocationID)')
plt.ylabel('WSM Score')
plt.title('WSM Welfare Score — Top 20 Zones\n(Green = Recommended)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('graph4_wsm_scores.png', dpi=150)
plt.show()
print('Graph 4 saved')

Graph 4 saved


In [15]:
# ── Graph 5: Baseline Comparison Bar Chart ────────────────────────────────────
labels   = ['Random\nZone', 'Nearest\nZone', 'Greedy\nDemand', 'Our System\n(WSM)']
revenues = [random_rev, nearest_rev, greedy_rev, our_rev]
colors5  = ['tomato', 'orange', 'goldenrod', 'green']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, revenues, color=colors5, edgecolor='black', linewidth=0.5, width=0.5)
for bar, val in zip(bars, revenues):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(revenues)*0.01,
            f'${val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Avg Per-Driver Revenue / 15-min Window ($)')
ax.set_title('Baseline Comparison\n(Corrected Per-Driver Earnings)')
plt.tight_layout()
plt.savefig('graph5_baseline_comparison.png', dpi=150)
plt.show()
print('Graph 5 saved')

Graph 5 saved


In [16]:
# ── Graph 6: Zone Distribution from dbt ──────────────────────────────────────
con = duckdb.connect(r"C:\Users\pujit\DE PROJECT\pipeline.duckdb")
zone_counts_df = con.execute("""
    SELECT zone_id AS PULocationID, COUNT(*) AS trip_count
    FROM int_tlc_cleaned
    GROUP BY zone_id
    ORDER BY trip_count DESC
""").df()
con.close()

def gini(arr):
    arr = np.sort(arr.astype(float))
    n   = len(arr)
    return (2 * np.sum(np.arange(1, n+1) * arr) / (n * arr.sum())) - (n+1)/n

g = gini(zone_counts_df['trip_count'].values)

plt.figure(figsize=(10, 4))
zone_counts_df.head(40).plot(x='PULocationID', y='trip_count', kind='bar',
                              color='steelblue', edgecolor='black', linewidth=0.3, ax=plt.gca())
plt.xlabel('PULocationID (Top 40 Zones)')
plt.ylabel('Trip Count')
plt.title(f'Zone Trip Distribution (Gini={g:.3f})')
plt.xticks(fontsize=7, rotation=90)
plt.tight_layout()
plt.savefig('graph6_zone_distribution.png', dpi=150)
plt.show()
print('Graph 6 saved')

Graph 6 saved


In [17]:
# ── Graph 7: Revenue vs Idle Penalty per Top 15 Zones ─────────────────────────
top15 = zones_sorted.head(15).copy()
top15['revenue_score'] = top15['predicted_demand'] * top15['avg_fare'] / top15['avg_drivers']
top15['idle_score']    = beta / (top15['demand_density'] + 1e-6)

x     = np.arange(len(top15))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, top15['revenue_score'], width,
       label='Per-Driver Revenue (α×R)', color='green', alpha=0.82, edgecolor='black', linewidth=0.4)
ax.bar(x + width/2, top15['idle_score'], width,
       label='Idle Penalty (β×I)', color='red', alpha=0.82, edgecolor='black', linewidth=0.4)
ax.set_xticks(x)
ax.set_xticklabels(top15['PULocationID'].astype(int), rotation=45, fontsize=8)
ax.set_xlabel('Zone (PULocationID)')
ax.set_ylabel('Score Component Value')
ax.set_title('Per-Driver Revenue vs Idle Penalty — Top 15 Zones')
ax.legend()
plt.tight_layout()
plt.savefig('graph7_revenue_vs_idle.png', dpi=150)
plt.show()
print('Graph 7 saved')

Graph 7 saved


## 11. Final Summary

In [15]:
print('\n' + '='*55)
print('           FINAL SUMMARY')
print('='*55)
print(f'  Dataset          : Jan–Mar 2025 NYC Yellow Taxi')
print(f'  Zones used       : {len(zones)} (official NYC 263-zone system)')
print(f'  R²               : {r2:.4f}')
print(f'  MAE              : {mae:.4f}')
print(f'  RMSE             : {rmse:.4f}')
print(f'  MAPE             : {mape:.2f}%')
print(f'  NRMSE            : {nrmse:.2f}%')
print(f'  Best zone        : Zone {int(best_zone["PULocationID"])}')
print(f'  WSM weights      : α={alpha}, β={beta}, γ={gamma}')
print(f'  Per-driver rev   : ${our_rev:.2f} / 15-min window')
print(f'  vs Random        : +{((our_rev-random_rev)/random_rev)*100:.1f}%')
print(f'  vs Nearest       : +{((our_rev-nearest_rev)/nearest_rev)*100:.1f}%')
print(f'  vs Greedy        : +{((our_rev-greedy_rev)/greedy_rev)*100:.1f}%')
print('='*55)


           FINAL SUMMARY
  Dataset          : Jan–Mar 2025 NYC Yellow Taxi
  Zones used       : 232 (official NYC 263-zone system)
  R²               : 0.9399
  MAE              : 2.6004
  RMSE             : 5.0757
  MAPE             : 41.05%
  NRMSE            : 1.97%
  Best zone        : Zone 132
  WSM weights      : α=0.6, β=0.2, γ=0.1
  Per-driver rev   : $53.58 / 15-min window
  vs Random        : +219.2%
  vs Nearest       : +28.6%
  vs Greedy        : +350.6%
